[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/27_vit_patch_solution.ipynb)

# 🟠 Solution: ViT Patch Embedding（参考版）

## 📋 题目描述

**难度：** Medium

实现 ViT 图像块嵌入（nn.Module）。

图像块嵌入将图像分割为固定大小的块，并将每个块投影为嵌入向量，形成 Vision Transformer 的输入序列。

**签名:** `PatchEmbedding(img_size, patch_size, in_channels, embed_dim)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 图像张量 (B, C, H, W)

**返回:** 图像块嵌入 (B, num_patches, embed_dim)

**约束:**
- `num_patches = (img_size / patch_size)^2`
- 将 `self.num_patches` 存储为属性
- 使用 `nn.Linear(C * P * P, embed_dim)` 投影

**提示：** `(B,C,H,W)` → reshape 为 `(B, n_h, P, n_w, P, C)` → permute → `(B, num_patches, C*P*P)` → `nn.Linear(C*P*P, embed_dim)`。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        # ---- Step 1: 保存参数 ----
        # patch_size: 每个图像块的边长，比如 16
        # num_patches: 图像块总数 = (img_size / patch_size)²
        # 例如 224×224 图像，patch_size=16 → (224/16)² = 14² = 196 个 patch
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        # ---- Step 2: 线性投影层 ----
        # 每个 patch 展平后维度 = C × P × P
        # 例如 RGB 图像 (C=3), patch_size=16 → 3×16×16 = 768
        # nn.Linear 将展平的 patch 投影到 embed_dim 维
        self.proj = nn.Linear(in_channels * patch_size * patch_size, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        n_h, n_w = H // p, W // p  # 行列各有多少个 patch

        # ---- Step 3: 切分图像为 patch ----
        # x shape: (B, C, H, W)
        # reshape: (B, C, n_h, p, n_w, p)
        #   C: 通道数
        #   n_h, n_w: patch 的行列数
        #   p, p: 每个 patch 的高和宽
        x = x.reshape(B, C, n_h, p, n_w, p)

        # ---- Step 4: 重排维度 ----
        # permute: (B, n_h, n_w, C, p, p)
        #   把 patch 坐标 (n_h, n_w) 放在前面
        #   通道和 patch 内像素放后面
        x = x.permute(0, 2, 4, 1, 3, 5)

        # ---- Step 5: 展平为序列 ----
        # reshape: (B, n_h*n_w, C*p*p)
        #   n_h*n_w = num_patches（序列长度）
        #   C*p*p = 每个 patch 的展平维度
        x = x.reshape(B, n_h * n_w, C * p * p)

        # ---- Step 6: 线性投影 ----
        # (B, num_patches, C*p*p) → (B, num_patches, embed_dim)
        return self.proj(x)


In [ ]:
pe = PatchEmbedding(img_size=32, patch_size=4, in_channels=3, embed_dim=64)
x = torch.randn(2, 3, 32, 32)
out = pe(x)
print(f'num_patches: {pe.num_patches}, Output: {out.shape}')


In [ ]:
from torch_judge import check
check('vit_patch')


## 📝 核心思路总结

1. **图像切分**：reshape + permute 将 (B,C,H,W) 切成 (B, num_patches, C*P*P)
2. **维度重排**：`(C,n_h,p,n_w,p)` → `(n_h,n_w,C,p,p)` 确保 patch 按行列顺序
3. **线性投影**：展平后的 patch 通过 Linear 映射到 embed_dim
4. **等价 Conv2d**：`nn.Conv2d(C, D, P, stride=P)` 效果相同
